In [9]:
import pandas as pd
import os
import re
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

In [10]:
# ==================================================
# KONFIGURASI
# ==================================================
file_excel = "./rekap_UBUM.xlsx"
folder_output = "hasil_split"

os.makedirs(folder_output, exist_ok=True)

# ==================================================
# BACA DATA
# ==================================================
df = pd.read_excel(file_excel)
df.columns = df.columns.astype(str).str.strip()
# ==================================================
# FILTER DATA: summary_category TIDAK KOSONG
# ==================================================
jumlah_awal = len(df)

kategori_dipakai = [
    "maps_kurang_lowongan_cocok_lokasi",
    "maps_kurang_lowongan_lokasi_lain",
    "maps_kurang_tanpa_lowongan",
]

jumlah_awal = len(df)

df = df[df["summary_category"].astype(str).str.strip().isin(kategori_dipakai)].copy()


print(f"Jumlah data awal       : {jumlah_awal:,}")
print(f"Jumlah data setelah filter: {len(df):,}")
print(f"Data yang tidak dipakai: {jumlah_awal - len(df):,}")

# ==================================================
# PEMETAAN KOLOM SUMBER KE FORMAT GROUND CHECK
# ==================================================
mapping_kolom = {
    "idsbr": "idsbr",
    "nama": "nama",
    "alamat": "alamat",
    "kd_prov": "kdprov",
    "kd_kab": "kdkab",
    "nmkab": "nama_kab",
    "kd_kec": "kdkec",
    "kd_desa": "kddesa",
    "flag_keberadaan": "flag_keberadaan",
    "flag_usaha": "flag_usaha",
    "maps": "maps",
    "maps_nama": "maps_nama",
    "maps_alamat_teks": "maps_alamat_teks",
    "website": "website",
}

kolom_tidak_ada = [kolom for kolom in mapping_kolom if kolom not in df.columns]

if kolom_tidak_ada:
    raise ValueError(f"Kolom sumber tidak ditemukan: {kolom_tidak_ada}")

hasil = df[list(mapping_kolom)].rename(columns=mapping_kolom).copy()

Jumlah data awal       : 4,856
Jumlah data setelah filter: 159
Data yang tidak dipakai: 4,697


In [11]:
# ==================================================
# TAMBAHKAN KOLOM GROUND CHECK KOSONG
# ==================================================
hasil["GROUND CHECK: Perusahaan/Usaha Ditemukan?"] = ""
hasil["Update kode keberadaan Usaha di FASIH"] = ""
hasil["Keterangan"] = ""

kolom_akhir = [
    "idsbr",
    "nama",
    "alamat",
    "kdprov",
    "kdkab",
    "nama_kab",
    "kdkec",
    "kddesa",
    "flag_keberadaan",
    "flag_usaha",
    "maps",
    "maps_nama",
    "maps_alamat_teks",
    "website",
    "GROUND CHECK: Perusahaan/Usaha Ditemukan?",
    "Update kode keberadaan Usaha di FASIH",
    "Keterangan",
]

hasil = hasil[kolom_akhir]

# ==================================================
# REFERENSI DROPDOWN
# ==================================================
pilihan_ground_check = ["Ya", "Tidak"]

pilihan_fasih = [
    "0 - tidak ditemukan",
    "1 - ditemukan (CAPI UB)/pindah dan dapat ditelusuri (UB)",
    "2 - baru",
    "3 - tutup (CAPI UB)",
    "4 - duplikat/ganda (CAPI UB)",
    "9 - tidak bersedia diwawancarai sampai akhir pendataan (CAPI UB)",
    "10 - ditemukan tetapi bukan cakupan survei (UB)",
    "11 - pindah dan tidak dapat ditelusuri (UB)",
    "12 - data diperoleh dari kantor pusat (CAPI UB)",
]

In [12]:
# ==================================================
# SPLIT PER KABUPATEN
# ==================================================
total_file = 0

for kd_kab, data_kab in hasil.groupby("kdkab", dropna=False):

    if pd.isna(kd_kab):
        nama_kab_file = "KODE_KAB_KOSONG"
    else:
        nama_kab_file = re.sub(r'[\\/:*?"<>|]', "_", str(kd_kab))

    path_output = os.path.join(folder_output, f"ground_check_{nama_kab_file}.xlsx")

    # Buat workbook; data dimulai pada baris ke-3
    with pd.ExcelWriter(path_output, engine="openpyxl") as writer:
        data_kab.to_excel(
            writer, sheet_name="Ground Check", index=False, header=False, startrow=2
        )

        ws = writer.sheets["Ground Check"]

        # ------------------------------------------
        # BARIS 1: NAMA KOLOM
        # BARIS 2: NOMOR KOLOM
        # BARIS 3: DATA
        # ------------------------------------------
        for col_idx, nama_kolom in enumerate(kolom_akhir, start=1):
            cell_header = ws.cell(row=1, column=col_idx)
            cell_header.value = nama_kolom

            cell_nomor = ws.cell(row=2, column=col_idx)
            cell_nomor.value = f"({col_idx})"

            cell_header.font = Font(bold=True, color="FFFFFF")
            cell_header.fill = PatternFill("solid", fgColor="1F4E78")
            cell_header.alignment = Alignment(
                horizontal="center", vertical="center", wrap_text=True
            )

            cell_nomor.font = Font(bold=True, italic=True)
            cell_nomor.fill = PatternFill("solid", fgColor="D9EAF7")
            cell_nomor.alignment = Alignment(horizontal="center", vertical="center")

        # ------------------------------------------
        # LEBAR KOLOM
        # ------------------------------------------
        lebar_kolom = {
            1: 16,  # idsbr
            2: 28,  # nama
            3: 45,  # alamat
            4: 10,  # kdprov
            5: 10,  # kdkab
            6: 22,  # nama_kab
            7: 10,  # kdkec
            8: 10,  # kddesa
            9: 16,  # flag_keberadaan
            10: 13,  # flag_usaha
            11: 32,  # maps
            12: 30,  # maps_nama
            13: 40,  # maps_alamat_teks
            14: 30,  # website
            15: 20,  # ground check
            16: 48,  # update kode FASIH
            17: 35,  # keterangan
        }

        for col_idx, lebar in lebar_kolom.items():
            ws.column_dimensions[get_column_letter(col_idx)].width = lebar

        # Tinggi header
        ws.row_dimensions[1].height = 48
        ws.row_dimensions[2].height = 24

        # ------------------------------------------
        # FORMAT DATA DAN WRAP TEXT
        # ------------------------------------------
        baris_awal_data = 3
        baris_akhir_data = ws.max_row

        for row_idx in range(baris_awal_data, baris_akhir_data + 1):
            # Semua sel: teks di atas dan wrap text
            for col_idx in range(1, 18):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.alignment = Alignment(vertical="top", wrap_text=True)

            # Alamat: perkiraan tinggi baris sesuai panjang teks
            alamat = ws.cell(row=row_idx, column=3).value
            if alamat:
                panjang_baris = max(1, (len(str(alamat)) + 44) // 45)
                ws.row_dimensions[row_idx].height = min(
                    max(30, panjang_baris * 15), 120
                )

        # ------------------------------------------
        # DROPDOWN GROUND CHECK: YA / TIDAK
        # ------------------------------------------
        # Daftar referensi disimpan di kolom tersembunyi U
        for idx, pilihan in enumerate(pilihan_ground_check, start=1):
            ws.cell(row=idx, column=21).value = pilihan  # U1 = Ya, U2 = Tidak

        dv_ground = DataValidation(
            type="list",
            formula1=f"=$U$1:$U${len(pilihan_ground_check)}",
            allow_blank=True,
        )
        dv_ground.error = "Silakan pilih Ya atau Tidak."
        dv_ground.errorTitle = "Pilihan tidak valid"
        dv_ground.prompt = "Pilih Ya atau Tidak."
        dv_ground.promptTitle = "Ground Check"
        ws.add_data_validation(dv_ground)

        if baris_akhir_data >= baris_awal_data:
            dv_ground.add(f"O{baris_awal_data}:O{baris_akhir_data}")

        # ------------------------------------------
        # DROPDOWN UPDATE KODE KEBERADAAN FASIH
        # ------------------------------------------
        for idx, pilihan in enumerate(pilihan_fasih, start=1):
            ws.cell(row=idx, column=22).value = pilihan  # V

        dv_fasih = DataValidation(
            type="list", formula1=f"=$V$1:$V${len(pilihan_fasih)}", allow_blank=True
        )
        dv_fasih.error = "Silakan pilih kode dari daftar."
        dv_fasih.errorTitle = "Kode tidak valid"
        dv_fasih.prompt = "Pilih kode dan keterangannya."
        dv_fasih.promptTitle = "Update kode FASIH"
        ws.add_data_validation(dv_fasih)

        if baris_akhir_data >= baris_awal_data:
            dv_fasih.add(f"P{baris_awal_data}:P{baris_akhir_data}")

        # Sembunyikan kolom referensi dropdown
        ws.column_dimensions["U"].hidden = True
        ws.column_dimensions["V"].hidden = True

        # ------------------------------------------
        # TAMPILAN SAAT FILE DIBUKA
        # ------------------------------------------
        ws.freeze_panes = "A3"
        ws.auto_filter.ref = (
            f"A2:Q{baris_akhir_data}" if baris_akhir_data >= 2 else "A1:Q1"
        )
        ws.sheet_view.showGridLines = False

        # Border tipis
        thin = Side(style="thin", color="D9E1F2")
        for row in ws.iter_rows(
            min_row=1, max_row=baris_akhir_data, min_col=1, max_col=17
        ):
            for cell in row:
                cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    total_file += 1
    print(f"Selesai: {nama_kab_file} ({len(data_kab)} baris)")

print("\n================================")
print("SPLIT EXCEL SELESAI")
print(f"Total file   : {total_file}")
print(f"Total data   : {len(hasil)}")
print(f"Folder hasil : {os.path.abspath(folder_output)}")
print("================================")

Selesai: 1601 (1 baris)
Selesai: 1602 (1 baris)
Selesai: 1603 (7 baris)
Selesai: 1606 (8 baris)
Selesai: 1607 (10 baris)
Selesai: 1608 (1 baris)
Selesai: 1609 (2 baris)
Selesai: 1610 (3 baris)
Selesai: 1613 (1 baris)
Selesai: 1671 (110 baris)
Selesai: 1672 (5 baris)
Selesai: 1673 (1 baris)
Selesai: 1674 (9 baris)

SPLIT EXCEL SELESAI
Total file   : 13
Total data   : 159
Folder hasil : d:\2026\Sensus Ekonomi\scrap usaha pusat\hasil_split
